# Tutorial 2: Load and Explore a SAHA CosMx Sample

This notebook walks through:
1. Finding a sample of interest via metadata
2. Loading the processed AnnData object from S3
3. Basic QC and cell type visualization
4. Spatial plotting of gene expression

## Setup

In [ ]:
# !pip install scanpy squidpy s3fs fsspec matplotlib seaborn

In [ ]:
import pandas as pd
import scanpy as sc
import squidpy as sq
import matplotlib.pyplot as plt
import seaborn as sns

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=100, frameon=False)

S3_OPTS = {"anon": True}

## Step 1: Find a sample

In [ ]:
samples = pd.read_parquet(
    "s3://saha-open-data/metadata/samples/",
    storage_options=S3_OPTS,
)

# Pick a large, QC-passing colon sample on CosMx with the 6K panel
candidates = samples.query(
    "organ == 'colon' and platform == 'cosmx' "
    "and qc_pass == True and panel_plex == 6000"
).sort_values("n_cells", ascending=False)

print(f"Found {len(candidates)} candidates")
candidates[["sample_id", "run_id", "n_cells", "n_fovs", "s3_processed_path"]].head(5)

In [ ]:
row = candidates.iloc[0]
print(f"Selected: {row['sample_id']}")
print(f"  organ:   {row['organ']}")
print(f"  n_cells: {row['n_cells']:,}")
print(f"  n_fovs:  {row['n_fovs']}")
print(f"  S3 path: {row['s3_processed_path']}")

## Step 2: Load the AnnData object

In [ ]:
# Stream directly from S3 — no local download needed for inspection
# backed="r" keeps data on disk until needed
adata = sc.read_h5ad(
    row["s3_processed_path"],
    storage_options=S3_OPTS,
    backed="r",
)
print(adata)

In [ ]:
# For interactive analysis, load fully into memory
adata = adata.to_memory()
print(adata.obs.columns.tolist())

## Step 3: Basic QC

In [ ]:
sc.pp.calculate_qc_metrics(
    adata, percent_top=None, log1p=False, inplace=True
)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].hist(adata.obs["total_counts"], bins=80, color="steelblue", edgecolor="none")
axes[0].set_xlabel("Total counts per cell")
axes[0].set_ylabel("# cells")
axes[0].set_title("Transcript counts")

axes[1].hist(adata.obs["n_genes_by_counts"], bins=80, color="coral", edgecolor="none")
axes[1].set_xlabel("Genes detected per cell")
axes[1].set_title("Genes per cell")

# Negative controls (if available)
neg_genes = [g for g in adata.var_names if g.startswith("NegPrb")]
if neg_genes:
    neg_counts = adata[:, neg_genes].X.sum(axis=1)
    axes[2].hist(neg_counts, bins=50, color="gray", edgecolor="none")
    axes[2].set_xlabel("Negative probe counts per cell")
    axes[2].set_title("Negative controls")
else:
    axes[2].text(0.5, 0.5, "No neg. probes", ha="center", va="center",
                  transform=axes[2].transAxes)

plt.tight_layout()
plt.show()

print(f"Median counts:        {adata.obs['total_counts'].median():.0f}")
print(f"Median genes/cell:    {adata.obs['n_genes_by_counts'].median():.0f}")

## Step 4: Cell type UMAP

In [ ]:
# If UMAP is already in adata.obsm, just plot it
if "X_umap" in adata.obsm:
    sc.pl.umap(
        adata,
        color="cell_type",
        legend_loc="on data",
        legend_fontsize=8,
        title=row["sample_id"],
        frameon=False,
    )
else:
    print("UMAP not precomputed — running from scratch ...")
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    sc.pp.highly_variable_genes(adata, n_top_genes=2000)
    sc.pp.pca(adata, n_comps=30)
    sc.pp.neighbors(adata, n_neighbors=15)
    sc.tl.umap(adata)
    sc.pl.umap(adata, color="cell_type", frameon=False)

## Step 5: Spatial cell type map

In [ ]:
# Spatial coordinates are stored in adata.obsm["spatial"] as (x, y) in microns
if "spatial" in adata.obsm:
    fig, ax = plt.subplots(1, 1, figsize=(10, 8))

    cell_types = adata.obs["cell_type"].cat.categories if hasattr(
        adata.obs["cell_type"], "cat"
    ) else adata.obs["cell_type"].unique()

    palette = sns.color_palette("tab20", len(cell_types))
    color_map = dict(zip(cell_types, palette))

    xy = adata.obsm["spatial"]
    colors = adata.obs["cell_type"].map(color_map)

    ax.scatter(xy[:, 0], xy[:, 1], c=colors, s=1, alpha=0.6, linewidths=0)
    ax.set_aspect("equal")
    ax.axis("off")
    ax.set_title(f"{row['sample_id']} — cell types in tissue", fontsize=12)

    # Legend
    from matplotlib.patches import Patch
    handles = [Patch(color=color_map[ct], label=ct) for ct in cell_types]
    ax.legend(handles=handles, bbox_to_anchor=(1.01, 1), loc="upper left",
               fontsize=7, frameon=False)

    plt.tight_layout()
    plt.show()
else:
    print("No spatial coordinates found in adata.obsm['spatial']")

## Step 6: Spatial gene expression

In [ ]:
# Pick a few marker genes to visualize spatially
marker_genes = [g for g in ["EPCAM", "CD3E", "CD68", "MKI67", "COL1A1"]
                if g in adata.var_names]

if marker_genes and "spatial" in adata.obsm:
    sq.pl.spatial_scatter(
        adata,
        color=marker_genes,
        shape=None,
        size=2,
        img=False,
        ncols=len(marker_genes),
        figsize=(4 * len(marker_genes), 4),
    )
else:
    print(f"Marker genes in panel: {marker_genes}")

## Step 7: Cell type composition

In [ ]:
ct_counts = adata.obs["cell_type"].value_counts()

fig, ax = plt.subplots(figsize=(6, 5))
ct_counts.plot(kind="barh", ax=ax, color="steelblue")
ax.set_xlabel("# cells")
ax.set_title(f"Cell type composition\n{row['sample_id']}")
plt.tight_layout()
plt.show()

print(ct_counts.to_string())

## Step 8: Neighborhood enrichment (spatial statistics)

In [ ]:
if "spatial" in adata.obsm and "cell_type" in adata.obs.columns:
    # Build spatial neighbors graph
    sq.gr.spatial_neighbors(adata, coord_type="generic", delaunay=True)

    # Neighborhood enrichment score
    sq.gr.nhood_enrichment(adata, cluster_key="cell_type")
    sq.pl.nhood_enrichment(adata, cluster_key="cell_type",
                            title="Neighborhood enrichment",
                            figsize=(8, 6))

## Next steps

- **Tutorial 03** — Cross-platform comparison (CosMx + Xenium) of the same organ
- Load raw CosMx flat files using `spatialdata-io` for FOV-level analysis
- See [FILE_FORMATS.md](../docs/FILE_FORMATS.md) for raw data format details